# Turbofan RUL Tahmini — Bölüm 2
## Preprocessing & Feature Engineering, Deney Tasarımı (GroupKFold) ve Baseline (Aşama 5–7)

Bu notebook Bölüm 1'deki veri kalitesi kararlarını uygular, **sızıntısız bir scikit-learn Pipeline** kurar, **motor bazlı GroupKFold** deney tasarımını tanımlar ve baseline modelleri değerlendirir.

> ⚠️ Yalnızca `train_FD001.txt` kullanılır. `test_FD001.txt` ve `RUL_FD001.txt` Aşama 12'ye kadar açılmaz.

**Bu bölümde alınan kararların özeti** (gerekçeleri ilgili başlıklarda):

| Karar | Seçim | Kanıt |
|---|---|---|
| Kullanılan özellikler | 14 bilgi taşıyan sensör | Bölüm 1, Aşama 4.3 |
| Hedef etiketleme | Parçalı doğrusal RUL, üst sınır **125** | 6.5 — cap deneyi |
| Feature engineering | Motor bazlı, yalnızca geçmişe bakan kayan **ortalama + std + eğim**, pencere **30** | 6.4 — özellik deneyi |
| Ölçekleme | `StandardScaler`, Pipeline içinde her fold'da yeniden fit | 5.5 |
| Validation | `GroupKFold(n_splits=5)`, gruplar = motor | 6.1–6.2 — sızıntı deneyi |
| Birincil validation metriği | Bakım karar ufkunda (gerçek RUL ≤ 150) test-benzeri RMSE | 6.3 |
| Baseline | `DummyRegressor(mean)` + **Linear Regression (ham 14 sensör)** | Aşama 7 |

## 0. Kurulum
`rul_utils.py` dosyası tüm notebook'larda ve Streamlit uygulamasında **aynı** dönüşüm ve metrik kodunun kullanılmasını sağlar (tekrarlanabilirlik + eğitim/servis tutarlılığı).

In [ ]:
%%writefile rul_utils.py
"""rul_utils.py — NASA C-MAPSS RUL projesi için ortak yardımcılar.
Notebook'lar ve Streamlit uygulaması aynı kodu kullanır (tekrarlanabilirlik)."""
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INDEX_COLS = ["unit", "cycle"]
SETTING_COLS = ["op1", "op2", "op3"]
SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
COLS = INDEX_COLS + SETTING_COLS + SENSOR_COLS
FEATURES_14 = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12",
               "s13", "s14", "s15", "s17", "s20", "s21"]


def load_cmapss(path):
    """Boşlukla ayrılmış, başlıksız C-MAPSS dosyasını okur."""
    df = pd.read_csv(path, sep=r"\s+", header=None).iloc[:, :26]
    df.columns = COLS
    return df.sort_values(["unit", "cycle"]).reset_index(drop=True)


def add_rul(df):
    """Eğitim verisi için gerçek (kırpılmamış) RUL: motorun son döngüsü - mevcut döngü."""
    df = df.copy()
    df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    return df


def _rolling_slope(x, window):
    """Geçmiş `window` döngüdeki doğrusal eğim (yalnızca geçmişe bakar)."""
    s = pd.Series(np.asarray(x, dtype=float))
    t = pd.Series(np.arange(len(s), dtype=float))
    cov = s.rolling(window, min_periods=2).cov(t)
    var = t.rolling(window, min_periods=2).var()
    return (cov / var).fillna(0.0).to_numpy()


class RollingFeatures(BaseEstimator, TransformerMixin):
    """Motor bazında, yalnızca GEÇMİŞ döngüleri kullanan kayan pencere özellikleri.

    - Girdi: 'unit', 'cycle' ve sensör sütunlarını içeren DataFrame (motor içinde döngü sırasına göre).
    - Çıktı: ham sensörler (+ isteğe bağlı) kayan ortalama, standart sapma ve eğim.
    - 'unit' ve 'cycle' çıktıya ALINMAZ (özellik değil, yalnızca gruplama/sıra bilgisi).
    - fit() hiçbir şey öğrenmez (stateless): bir satırın özellikleri yalnızca kendi motorunun
      o ana kadarki kayıtlarından hesaplanır → başka motorlardan veya gelecekten bilgi sızmaz.
    """

    def __init__(self, sensors=tuple(FEATURES_14), window=10, stats=("mean", "std", "slope"), keep_raw=True):
        self.sensors = sensors
        self.window = window
        self.stats = stats
        self.keep_raw = keep_raw

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        if not X.groupby("unit")["cycle"].apply(lambda c: c.is_monotonic_increasing).all():
            raise ValueError("Her motorun satırları döngü sırasına göre sıralı olmalı.")
        sensors, w = list(self.sensors), int(self.window)
        g = X.groupby("unit", sort=False)
        parts = []
        if self.keep_raw:
            parts.append(X[sensors].reset_index(drop=True))
        if "mean" in self.stats:
            m = g[sensors].transform(lambda c: c.rolling(w, min_periods=1).mean())
            parts.append(m.add_suffix(f"_mean{w}").reset_index(drop=True))
        if "std" in self.stats:
            sd = g[sensors].transform(lambda c: c.rolling(w, min_periods=2).std()).fillna(0.0)
            parts.append(sd.add_suffix(f"_std{w}").reset_index(drop=True))
        if "slope" in self.stats:
            sl = g[sensors].transform(lambda c: _rolling_slope(c, w))
            parts.append(sl.add_suffix(f"_slope{w}").reset_index(drop=True))
        out = pd.concat(parts, axis=1)
        out.index = X.index
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.sensors) if self.keep_raw else []
        for st in ("mean", "std", "slope"):
            if st in self.stats:
                names += [f"{s}_{st}{self.window}" for s in self.sensors]
        return np.array(names)


def nasa_score(y_true, y_pred):
    """PHM08 skor fonksiyonu. d = tahmin - gerçek.
    Geç tahmin (d > 0, ömrü fazla tahmin etmek) daha ağır cezalandırılır."""
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1)))


def regression_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}R2": float(r2_score(y_true, y_pred)),
        f"{prefix}NASA": nasa_score(y_true, y_pred),
    }


def test_like_rows(df, idx, n_cuts=10, min_cycle=31, max_rul=150, seed=42):
    """Validation motorlarından resmi test protokolünü taklit eden satırlar seçer:
    her motor için rastgele `n_cuts` kesim noktası; kesim anındaki GERÇEK RUL ile değerlendirilir.
    - min_cycle: kayan pencerelerin dolması için gereken minimum geçmiş.
    - max_rul: bakım karar ufku. Kesim noktaları gerçek RUL <= max_rul olan döngülerden seçilir
      (EDA: bozulma ömrün ikinci yarısında görünür hale geliyor; daha erken döngülerde RUL
      sensörlerden ayırt edilemez ve bakım kararı açısından önemsizdir)."""
    rng = np.random.default_rng(seed)
    sub = df.loc[idx]
    chosen = []
    for _, d in sub.groupby("unit"):
        cand = d.index[(d["cycle"] >= min(min_cycle, d["cycle"].max())) & (d["RUL"] <= max_rul)]
        chosen.extend(rng.choice(cand, size=min(n_cuts, len(cand)), replace=False))
    return np.sort(np.array(chosen))


def cv_evaluate(model, df, cap, cv, feature_cols=None, n_cuts=10, max_rul=150, seed=42, name=None):
    """Motor bazlı CV ile değerlendirme.
    - Eğitim hedefi: RUL üst sınırla kırpılır (cap=None ise kırpılmaz).
    - 'clip_' metrikleri: validation motorlarının tüm döngüleri, kırpılmış hedefe göre.
    - 'test_' metrikleri: validation motorlarında rastgele kesim noktaları, GERÇEK RUL'a göre
      ve bakım karar ufku (gerçek RUL <= max_rul) içinde → model ve cap seçiminde birincil ölçüt.
    """
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_true = df["RUL"].to_numpy()
    y_fit = y_true if cap is None else np.minimum(y_true, cap)
    rows = []
    for k, (tr, va) in enumerate(cv.split(X, y_fit, groups=df["unit"])):
        m = clone(model)
        t0 = time.perf_counter()
        m.fit(X.iloc[tr], y_fit[tr])
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        p = np.clip(m.predict(X.iloc[va]), 0, None)
        pred_s = time.perf_counter() - t0
        res = {"fold": k + 1, "fit_s": fit_s, "pred_s": pred_s}
        res.update(regression_metrics(y_fit[va], p, "clip_"))
        cut = test_like_rows(df, df.index[va], n_cuts=n_cuts, max_rul=max_rul, seed=seed + k)
        pos = pd.Series(np.arange(len(va)), index=df.index[va]).loc[cut].to_numpy()
        res.update(regression_metrics(y_true[cut], p[pos], "test_"))
        rows.append(res)
    out = pd.DataFrame(rows)
    if name is not None:
        out.insert(0, "model", name)
    return out


def summarize(cv_df, by="model"):
    cols = ["test_RMSE", "test_MAE", "test_R2", "test_NASA", "clip_RMSE", "fit_s"]
    agg = cv_df.groupby(by)[cols].agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    return agg.sort_values("test_RMSE_mean")

In [ ]:
import warnings, json, sys, platform
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from rul_utils import *

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

df = add_rul(load_cmapss("train_FD001.txt"))
print("Eğitim verisi:", df.shape, "| motor:", df["unit"].nunique())

## Aşama 5 — Data Preprocessing ve Feature Engineering
### 5.1 Hedef değişken: parçalı doğrusal RUL

In [ ]:
CAP = 125
u = df[df["unit"] == 1]
plt.figure(figsize=(9, 4))
plt.plot(u["cycle"], u["RUL"], label="Gerçek (doğrusal) RUL")
plt.plot(u["cycle"], u["RUL"].clip(upper=CAP), lw=2.5, label=f"Parçalı doğrusal RUL (üst sınır = {CAP})")
plt.xlabel("döngü"); plt.ylabel("RUL"); plt.title("Motor 1 — hedef etiketleme"); plt.legend(); plt.show()
print(f"Üst sınırdan etkilenen satır oranı: %{(df['RUL'] > CAP).mean()*100:.1f}")

**Gerekçe:** Bölüm 1'deki 3.5 analizi, ömrün ilk ~%40'ında sensörlerin neredeyse değişmediğini gösterdi. Bu dönemde "RUL = 300" ile "RUL = 150"yi ayırt edecek bilgi verinin içinde yok; model bunları ayırt etmeye zorlanırsa gürültüyü öğrenir. Parçalı doğrusal etiket şunu söyler: *"Motor sağlıklı bölgedeyken kalan ömür en az 125 döngüdür; bozulma başladıktan sonra doğrusal azalır."* Fiziksel olarak da anlamlıdır.

- Kırpma **yalnızca eğitim hedefine** uygulanır. Değerlendirmede (validation test-benzeri metrikleri ve Aşama 12) **gerçek RUL** kullanılır.
- 125 değeri literatürde yaygındır, ama burada da körü körüne alınmadı: 6.5'te 100–150 arası değerler ve kırpmasız seçenek **CV ile karşılaştırıldı**.

### 5.2 Özellik çıkarma (Bölüm 1 kararlarının uygulanması)

In [ ]:
DROP_COLS = ["op1", "op2", "op3", "s1", "s5", "s6", "s10", "s16", "s18", "s19"]
reason = {**{c: "tam sabit (tek değer)" for c in ["op3", "s1", "s5", "s10", "s16", "s18", "s19"]},
          "s6": "2 farklı değer, sensör kuantalaması", "op1": "tek çalışma koşulu, RUL ile ilişkisiz",
          "op2": "tek çalışma koşulu, RUL ile ilişkisiz"}
display(pd.DataFrame({"çıkarılan": DROP_COLS, "gerekçe": [reason[c] for c in DROP_COLS]}))
print("Kullanılan 14 sensör:", FEATURES_14)
print("Gruplama/sıra bilgisi (özellik DEĞİL):", INDEX_COLS)

**Neden `unit` ve `cycle` özellik değil?**
- `unit` bir kimlik numarasıdır; modele verilirse yeni motorlara genelleyemez.
- `cycle` eğitimde RUL ile çok güçlü ilişkili görünür (r = −0.74), ama bu bir **tuzaktır**: motorların ömürleri farklıdır ve test motorları ömürlerinin rastgele bir noktasında kesilir. Model "döngü sayısı yüksekse ömür azdır" kuralını ezberlerse uzun ömürlü motorlarda sistematik hata yapar. Bozulmayı sensörlerden öğrenmesini istiyoruz.

### 5.3 Eksik değer, encoding ve aykırı değer kararları

| İşlem | Karar | Gerekçe |
|---|---|---|
| Missing-value imputation | **Uygulanmadı** | Eğitim verisinde eksik değer yok (Bölüm 1, 4.1). Gereksiz bir imputer eklemek modele bilgi katmaz. Uygulamada kullanıcıdan gelebilecek eksik/hatalı girişler arayüzde doğrulanacak (Aşama 13). |
| Encoding | **Gerekmiyor** | Tüm özellikler sayısal; kategorik değişken yok. |
| Outlier temizleme | **Uygulanmadı** | Aykırı görünen değerler arızaya yakın döngülerden geliyor (RUL medyanı 2–6); bu, tahmin etmek istediğimiz bozulma sinyalinin kendisi. Silmek en değerli bilgiyi atmak olur. |

### 5.4 Feature engineering: motor bazlı kayan pencere özellikleri

`RollingFeatures` dönüştürücüsü her satır için, **o motorun yalnızca geçmiş `window` döngüsüne** bakarak üç istatistik üretir:

| Özellik | Ne yakalar? | Neden gerekli? |
|---|---|---|
| `*_mean30` — kayan ortalama | Gürültüden arındırılmış sensör seviyesi | Bölüm 1, 4.6: ham sinyal gürültülü, trend ancak yumuşatınca görünüyor |
| `*_std30` — kayan standart sapma | Sinyaldeki oynaklık | Bozulma ilerledikçe ölçümlerin dalgalanması da değişebilir |
| `*_slope30` — kayan eğim | Bozulmanın **hızı** | İki motor aynı sensör seviyesinde olabilir ama biri çok daha hızlı bozuluyorsa kalan ömrü daha kısadır |

**Sızıntıyı önleyen tasarım kararları:**
1. **Motor içinde hesaplanır** (`groupby("unit")`): bir motorun penceresi asla başka bir motorun verisine taşmaz.
2. **Yalnızca geçmişe bakar** (`rolling`, `center=False`): gelecekteki döngüler kullanılmaz. Gerçek hayatta da bakım kararını verirken yalnızca o ana kadarki veriyi biliriz; test setindeki kesilmiş motorlar da tam olarak bu durumdadır.
3. **Durumsuzdur (stateless)**: `fit()` hiçbir şey öğrenmez. Bu yüzden dönüşüm, ölçekleyiciden farklı olarak eğitim verisinden test verisine istatistik taşımaz.
4. **Pipeline'ın içindedir**: Streamlit uygulaması ham sensör kaydını alıp aynı dönüşümü otomatik uygular → eğitim ile kullanım arasında tutarsızlık (training–serving skew) olmaz.

In [ ]:
fe = RollingFeatures(window=30)
F1 = fe.transform(df[df["unit"] == 1][INDEX_COLS + FEATURES_14])
print("Özellik sayısı:", F1.shape[1], "(14 ham + 14 ortalama + 14 std + 14 eğim)")

u = df[df["unit"] == 1]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(u["cycle"], u["s11"], alpha=.4, label="ham s11"); axes[0].plot(u["cycle"], F1["s11_mean30"], lw=2, label="s11_mean30")
axes[1].plot(u["cycle"], F1["s11_std30"], c="tab:orange"); axes[2].plot(u["cycle"], F1["s11_slope30"], c="tab:green")
for ax, t in zip(axes, ["Ham sinyal ve kayan ortalama", "Kayan standart sapma", "Kayan eğim (bozulma hızı)"]):
    ax.set_title(f"Motor 1 — s11 (Ps30): {t}"); ax.set_xlabel("döngü")
axes[0].legend(); plt.tight_layout(); plt.show()

**Yorum:** Kayan eğim, ömrün başında sıfır civarında dalgalanırken bozulma başladığında belirgin şekilde pozitife geçiyor — sensör seviyesinden önce "bozulmanın başladığını" yakalayan bir sinyal.

In [ ]:
# Sızıntı kontrolü: motor 1'in özellikleri, veri setinin geri kalanı olsa da olmasa da aynı mı?
F_alone = RollingFeatures(window=30).transform(df[df["unit"] == 1][INDEX_COLS + FEATURES_14])
F_all = RollingFeatures(window=30).transform(df[INDEX_COLS + FEATURES_14]).loc[df["unit"] == 1]
print("Motor 1 özellikleri tek başına == tüm veriyle birlikte:", np.allclose(F_alone.values, F_all.values))

# Geleceğe bakmama kontrolü: ilk 100 döngü ile hesaplanan özellikler, tüm kayıtla hesaplananların ilk 100 satırıyla aynı mı?
F_trunc = RollingFeatures(window=30).transform(df[(df["unit"] == 1) & (df["cycle"] <= 100)][INDEX_COLS + FEATURES_14])
print("Kesilmiş kayıt (100 döngü) == tam kaydın ilk 100 satırı:", np.allclose(F_trunc.values, F_alone.iloc[:100].values))

**İki test de `True` döner:** özellikler başka motorlardan etkilenmiyor ve gelecek döngülerden bilgi almıyor. Bu, kesilmiş test motorlarında özelliklerin eğitimdekiyle birebir aynı şekilde hesaplanacağının kanıtıdır.

### 5.5 Ölçekleme

`StandardScaler` Pipeline içinde yer alır; her CV fold'unda **yalnızca o fold'un eğitim motorları** üzerinde `fit` edilir, validation motorlarına sadece `transform` uygulanır.

**Gerekçe:** Sensörlerin ölçekleri çok farklı (ör. `s15` ≈ 8.4, `s9` ≈ 9000). Linear Regression'ın tahminleri ölçekten etkilenmez, ama Aşama 8'de karşılaştırılacak Ridge, KNN, SVR ve MLP gibi modeller ölçeğe çok duyarlıdır. **Tüm modellerin aynı preprocessing'i** kullanması adil karşılaştırma için şarttır. Ağaç tabanlı modeller ölçekten etkilenmez; ölçeklemenin onlara zararı da yoktur. Outlier'ları silmediğimiz için `RobustScaler` da düşünülebilirdi; ancak buradaki uç değerler hatalı ölçüm değil, gerçek bozulma sinyali olduğu için onları "bastırmak" istemiyoruz.

### 5.6 Feature selection

- Sabit / bilgisiz 10 sütun **çıkarıldı** (5.2).
- `s9` ve `s14` çok yüksek korelasyonlu (r = 0.96). Şimdilik ikisi de **tutuldu**: ağaç tabanlı modeller çoklu doğrusallıktan etkilenmez; doğrusal modellerde ise etkisi Ridge (L2) ile Aşama 8'de kontrol edilecek. Özellik önemleri Aşama 12'de incelenip gerekirse sadeleştirme yapılacak.

### 5.7 Pipeline

In [ ]:
def make_pipeline(model, window=30, stats=("mean", "std", "slope")):
    """Ham sensör kaydı → kayan özellikler → ölçekleme → model.
    stats=() verilirse yalnızca ham 14 sensör kullanılır."""
    return Pipeline([
        ("features", RollingFeatures(window=window, stats=stats)),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

make_pipeline(LinearRegression())

**Data leakage kontrol listesi:**

| Risk | Önlem |
|---|---|
| Aynı motorun döngüleri hem eğitimde hem validation'da | `GroupKFold`, grup = motor (6.1) |
| Kayan pencerenin başka motora taşması | `groupby("unit")` (5.4 — test edildi) |
| Kayan pencerenin geleceği görmesi | Yalnızca geçmişe bakan `rolling` (5.4 — test edildi) |
| Ölçekleyicinin validation istatistiklerini öğrenmesi | `StandardScaler` Pipeline içinde, fold içinde fit |
| Kimlik / sıra bilgisinin modele girmesi | `unit`, `cycle` özellik değil (5.2) |
| Test setinin model seçiminde kullanılması | Test dosyaları Aşama 12'ye kadar okunmuyor |
| Hiperparametrelerin (pencere, cap) test setine göre seçilmesi | Tüm seçimler yalnızca eğitim verisinde CV ile (6.4–6.5) |

## Aşama 6 — Veri Bölme ve Deney Tasarımı
### 6.1 Neden GroupKFold?

Bölüm 1, 4.8'de gördük: her motor ~206 ardışık ve birbirine çok benzeyen satırdan oluşuyor. Satırları rastgele bölersek, validation'daki bir döngünün hemen öncesi ve sonrası eğitimde olur; model motoru "tanır" ve skor yapay olarak iyi çıkar. Gerçek kullanımda ise model **hiç görmediği motorlar** için tahmin yapacak.

**Karar:** `GroupKFold(n_splits=5)`, `groups = unit`.
- **5 fold:** Her fold'da 80 motor eğitim, 20 motor validation. 100 motorluk küçük bir filoda her motor bir kez validation'da yer alır; 10 fold'a göre daha az varyanslı fold skorları ve makul hesaplama süresi sağlar.
- **Ayrı bir hold-out validation seti ayrılmadı:** Resmi test seti zaten tamamen ayrı 100 motordan oluşuyor; eğitimden ayrıca motor ayırmak, zaten küçük olan eğitim filosunu gereksiz yere küçültürdü. CV hem model seçimi hem tuning için kullanılacak; final rapor resmi test setinden gelecek.
- **Tekrarlanabilirlik:** `GroupKFold` deterministiktir (rastgelelik içermez); aynı veriyle her çalıştırmada aynı fold'lar oluşur. Modellerin kendi rastgeleliği `random_state=42` ile sabitlenir.

In [ ]:
cv = GroupKFold(n_splits=5)
rows = []
for k, (tr, va) in enumerate(cv.split(df, groups=df["unit"])):
    tr_u, va_u = set(df.iloc[tr]["unit"]), set(df.iloc[va]["unit"])
    rows.append({"fold": k + 1, "eğitim_motor": len(tr_u), "val_motor": len(va_u), "eğitim_satır": len(tr),
                 "val_satır": len(va), "ortak_motor": len(tr_u & va_u),
                 "val_ömür_ort": df.iloc[va].groupby("unit")["cycle"].max().mean().round(1)})
display(pd.DataFrame(rows))

**Bulgu:** Fold'lar dengeli (her birinde 20 validation motoru, ~4.120 satır) ve hiçbir motor hem eğitimde hem validation'da yer almıyor (`ortak_motor = 0`).

### 6.2 Sızıntının etkisini ölçmek: KFold vs GroupKFold

In [ ]:
F = RollingFeatures(window=30).transform(df[INDEX_COLS + FEATURES_14])
y = df["RUL"].clip(upper=CAP)
demo = Pipeline([("scaler", StandardScaler()), ("model", HistGradientBoostingRegressor(random_state=SEED))])

leak = {}
for name, splitter in [("KFold — satır bazlı, karıştırılmış", KFold(5, shuffle=True, random_state=SEED)),
                       ("GroupKFold — motor bazlı", cv)]:
    s = -cross_val_score(demo, F, y, cv=splitter, groups=df["unit"] if "Group" in name else None,
                         scoring="neg_root_mean_squared_error")
    leak[name] = s
    print(f"{name:38s} RMSE = {s.mean():.2f} ± {s.std():.2f}")
gap = (leak["GroupKFold — motor bazlı"].mean() - leak["KFold — satır bazlı, karıştırılmış"].mean()) / leak["GroupKFold — motor bazlı"].mean()
print(f"\nSatır bazlı bölme hatayı %{gap*100:.0f} daha düşük (yani yanıltıcı biçimde iyimser) gösteriyor.")

**Yorum:** Aynı model ve aynı özelliklerle, sadece bölme yöntemini değiştirmek RMSE'yi ~13.2'den ~8.2'ye düşürüyor. Bu ~%38'lik "iyileşme" tamamen sahte: model, validation'daki döngülerin komşularını eğitimde gördüğü için motoru tanıyor. Satır bazlı bölme kullansaydık, modelimizin literatürdeki derin öğrenme modellerinden çok daha iyi olduğunu sanacaktık. **Bu deney, GroupKFold seçiminin kanıtıdır.**

### 6.3 Değerlendirme protokolü: iki validation metriği

Resmi test setinde her motor ömrünün rastgele bir noktasında kesilir ve tahmin yalnızca **son döngü** için, **gerçek (kırpılmamış) RUL**'a göre değerlendirilir. Validation'ı bu protokole benzetmek için `cv_evaluate` her fold'da iki tür metrik hesaplar:

| Metrik | Nasıl hesaplanır? | Ne için? |
|---|---|---|
| `clip_*` | Validation motorlarının **tüm döngüleri**, **kırpılmış** hedefe göre | Modelin eğitim hedefini ne kadar iyi öğrendiği |
| **`test_*`** (birincil) | Her validation motorundan **10 rastgele kesim noktası**, kesim anındaki **gerçek RUL**'a göre | Resmi test protokolünün taklidi → model, pencere ve cap seçimi |

**Test-benzeri kesim noktaları için iki tasarım kararı:**
- **`min_cycle = 31`:** Kesim noktası en az 31. döngüde seçilir, böylece 30 döngülük pencere bir kez tamamen dolmuş olur.
- **`max_rul = 150` (bakım karar ufku):** Kesim noktaları gerçek RUL'u 150 ve altında olan döngülerden seçilir. Gerekçe: (1) EDA'ya göre bozulma ömrün ikinci yarısında görünür hale geliyor (medyan ömür 199 döngü); daha önceki döngülerde RUL'u sensörlerden ayırt etmek mümkün değil. (2) Bakım planlaması açısından önemli olan, motorun arızaya yaklaştığı dönemde doğru tahmin yapmak; "300 döngü mü 250 döngü mü kaldı" sorusu bir karar değiştirmez. Bu sınır **test verisine bakılmadan**, yalnızca eğitim verisinin EDA'sı ve problem tanımından belirlendi.

**Raporlanan metrikler:** RMSE (birincil, literatürle karşılaştırılabilir), MAE, R², NASA skoru (geç tahminleri cezalandırır) ve eğitim süresi. Her biri **5 fold'un ortalaması ± standart sapması** olarak verilir.

### 6.4 Feature engineering deneyi: hangi özellikler ve hangi pencere?

Özellik setleri iki farklı model ailesiyle test edildi: **Linear Regression** (baseline) ve **HistGradientBoosting** (varsayılan parametrelerle, yalnızca *duyarlılık kontrolü* olarak — model seçimi Aşama 8'de yapılacak). Gerekçe: feature engineering kararı sadece doğrusal bir modelin bakış açısına göre verilirse, doğrusal olmayan modellerin yararlanabileceği özellikler gözden kaçabilir.

In [ ]:
COLS_SHOW = ["test_RMSE_mean", "test_RMSE_std", "test_MAE_mean", "test_NASA_mean", "clip_RMSE_mean", "fit_s_mean"]
fe_res = []
configs = [("ham 14 sensör", dict(stats=())),
           ("+ ortalama (w=30)", dict(window=30, stats=("mean",))),
           ("+ ort + std + eğim (w=10)", dict(window=10)),
           ("+ ort + std + eğim (w=20)", dict(window=20)),
           ("+ ort + std + eğim (w=30)", dict(window=30))]
for model_name, model in [("LR", LinearRegression()), ("HGB", HistGradientBoostingRegressor(random_state=SEED))]:
    for name, kw in configs:
        r = cv_evaluate(make_pipeline(model, **kw), df, CAP, cv, name=f"{model_name} | {name}")
        fe_res.append(r)
fe_res = pd.concat(fe_res)
fe_sum = summarize(fe_res)[COLS_SHOW]
display(fe_sum.round(2))

In [ ]:
plot_df = fe_sum.reset_index()
plot_df["model_ailesi"] = plot_df["model"].str.split(" \\| ").str[0]
plot_df["özellik_seti"] = plot_df["model"].str.split(" \\| ").str[1]
order = [c[0] for c in configs]
plt.figure(figsize=(11, 4.5))
sns.barplot(data=plot_df, x="özellik_seti", y="test_RMSE_mean", hue="model_ailesi", order=order)
plt.ylabel("Test-benzeri RMSE (CV ortalaması)"); plt.xlabel(""); plt.xticks(rotation=15)
plt.title("Feature engineering'in etkisi (düşük daha iyi)"); plt.show()

**Yorum ve karar:**
- Linear Regression'da kayan özellikler RMSE'yi ~23.2'den ~21.3'e düşürüyor (~%8). Sınırlı bir kazanç, çünkü doğrusal model bozulmanın doğrusal olmayan (hızlanan) yapısını tam yakalayamıyor.
- HistGradientBoosting'de etki çok daha büyük: ~21.8 → ~16.0 (~%26). Ham sensörlerle iki model neredeyse aynı seviyedeyken, zengin özelliklerle ağaç modeli belirgin şekilde öne geçiyor.
- Sadece kayan ortalama yeterli değil; **std ve eğim** önemli ek bilgi taşıyor (özellikle bozulmanın hızı).
- Pencere büyüdükçe performans iyileşiyor (10 → 20 → 30). Daha uzun pencereler gürültüyü daha iyi bastırıyor.

**Karar:** Ham sensörler + kayan ortalama/std/eğim, **pencere = 30** (56 özellik). Pencere uzunluğu bir hiperparametredir; Aşama 10'da model hiperparametreleriyle birlikte yeniden aranacak (30'dan uzun pencereler de denenecek).

### 6.5 RUL üst sınırı (cap) deneyi

In [ ]:
cap_res = []
for cap in [None, 100, 115, 125, 135, 150]:
    for model_name, model in [("LR", LinearRegression()), ("HGB", HistGradientBoostingRegressor(random_state=SEED))]:
        r = cv_evaluate(make_pipeline(model, window=30), df, cap, cv, name=f"{model_name} | cap={cap}")
        r["model_ailesi"], r["cap"] = model_name, "yok" if cap is None else cap
        cap_res.append(r)
cap_res = pd.concat(cap_res)
cap_sum = cap_res.groupby(["model_ailesi", "cap"])[["test_RMSE", "test_NASA", "clip_RMSE"]].mean()
display(cap_sum.unstack(0).round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
order = ["yok", 100, 115, 125, 135, 150]
for fam, g in cap_res.groupby("model_ailesi"):
    m = g.groupby("cap")["test_RMSE"].agg(["mean", "std"]).reindex(order)
    ax[0].errorbar([str(o) for o in order], m["mean"], yerr=m["std"], marker="o", capsize=4, label=fam)
    n = g.groupby("cap")["test_NASA"].mean().reindex(order)
    ax[1].plot([str(o) for o in order], n, marker="o", label=fam)
ax[0].set_title("Test-benzeri RMSE (ort ± std)"); ax[1].set_title("NASA skoru (log ölçek)"); ax[1].set_yscale("log")
for a in ax: a.set_xlabel("RUL üst sınırı"); a.legend()
plt.tight_layout(); plt.show()

**Yorum ve karar:**
- **Kırpmasız eğitim en kötü sonucu veriyor** (HGB ~24.4, LR ~32.5 RMSE; NASA skoru 40–70 kat daha yüksek). Model ömrün başındaki ayırt edilemeyen yüksek RUL değerlerini öğrenmeye çalışırken bozulma bölgesinde de kötüleşiyor. Bu, 5.1'deki gerekçeyi doğruluyor.
- **115 ve 125 neredeyse eşit** (HGB: ~16.2 vs ~16.0; LR: ~21.2 vs ~21.3); farklar fold standart sapmasının (~1–2) içinde. 100 fazla agresif (gerçek RUL'u 100–150 arası olan motorları sistematik olarak eksik tahmin ediyor), 135 ve 150 ise bozulma öncesi dönemden gürültü almaya başlıyor.
- **Karar: cap = 125.** CV'de en iyi ya da en iyiye istatistiksel olarak eşit değer; ek olarak literatürdeki standart değer olduğu için Aşama 12'de yayınlanmış sonuçlarla **doğrudan ve adil karşılaştırma** imkânı veriyor. 115'in NASA skoru biraz daha düşük (daha temkinli tahminler); bu alternatif raporda not edilecek.

### 6.6 Test setinin korunması

- Bu notebook'ta `test_FD001.txt` ve `RUL_FD001.txt` **hiç okunmadı**.
- Özellik seti, pencere, cap ve karar ufku seçimlerinin tamamı eğitim verisi üzerinde CV ile yapıldı.
- Aşama 8–11'deki model seçimi ve tuning de aynı `cv` nesnesi ve aynı `cv_evaluate` fonksiyonuyla yapılacak → tüm modeller **aynı fold'lar ve aynı kesim noktaları** üzerinde karşılaştırılacak.
- Resmi test seti Aşama 12'de **yalnızca bir kez**, final model için açılacak.

## Aşama 7 — Baseline Model
### 7.1 Baseline seçimi ve gerekçesi

| Baseline | Rolü | Neden? |
|---|---|---|
| `DummyRegressor(strategy="mean")` | **Alt sınır** | Sensörlere hiç bakmadan herkese ortalama RUL'u tahmin eder. Bir modelin "bir şey öğrendiğini" söyleyebilmek için bunu belirgin şekilde geçmesi gerekir. |
| `DummyRegressor(strategy="median")` | Alt sınır (alternatif) | Çarpık hedeflerde medyan bazen daha iyi bir sabit tahmindir; karşılaştırma için eklendi. |
| **Linear Regression — ham 14 sensör** | **Ana baseline** | (1) Hiperparametresi yok → tuning avantajı olmadan "saf" başlangıç performansı. (2) Tamamen yorumlanabilir (katsayılar). (3) Literatürde bu veri seti için yaygın kullanılan referans; raporladığı ~22 RMSE ile bizim sonucumuzu karşılaştırarak **doğrulama protokolümüzün doğruluğunu da kontrol etmiş oluyoruz**. (4) Eğitimi milisaniyeler sürer. |
| Linear Regression — kayan özellikler | Feature engineering'in doğrusal modele katkısı | Aşama 5'teki özellik mühendisliğinin değerini, model karmaşıklığından bağımsız olarak gösterir. |

**Neden Linear Regression ve ham sensörler "ana" baseline?** Baseline'ın amacı, bu problemde "makul ama basit" bir yöntemin nereye kadar gidebildiğini göstermek. Feature engineering ve doğrusal olmayan modellerin her biri, bu referansa göre ne kadar katkı sağladığıyla değerlendirilecek. Ayrıca alan bilgisine dayalı basit bir sezgisel kural ("ortalama ömür − mevcut döngü") düşünülebilirdi; ancak bu kural `cycle` bilgisine dayanır ve 5.2'de açıklanan nedenlerle (farklı motor ömürleri, rastgele kesim noktaları) yanıltıcı olacağı için baseline olarak kullanılmadı.

Tüm baseline'lar **aynı Pipeline, aynı cap (125), aynı GroupKFold fold'ları ve aynı test-benzeri kesim noktaları** ile değerlendirildi.

In [ ]:
baselines = {
    "Dummy (ortalama)": make_pipeline(DummyRegressor(strategy="mean"), stats=()),
    "Dummy (medyan)": make_pipeline(DummyRegressor(strategy="median"), stats=()),
    "LinearRegression — ham 14 sensör": make_pipeline(LinearRegression(), stats=()),
    "LinearRegression — kayan özellikler (w=30)": make_pipeline(LinearRegression(), window=30),
}
base_res = pd.concat([cv_evaluate(p, df, CAP, cv, name=n) for n, p in baselines.items()])
base_sum = summarize(base_res)
display(base_sum[["test_RMSE_mean", "test_RMSE_std", "test_MAE_mean", "test_R2_mean", "test_NASA_mean", "clip_RMSE_mean", "fit_s_mean"]].round(3))

### 7.2 Fold bazında sonuçlar — ana baseline

In [ ]:
display(base_res[base_res["model"] == "LinearRegression — ham 14 sensör"][["fold", "test_RMSE", "test_MAE", "test_R2", "test_NASA", "clip_RMSE"]].round(2))

### 7.3 Görsel değerlendirme (out-of-fold tahminler)

In [ ]:
def oof_predictions(pipe, df, cap, cv):
    """Her satır için, o satırın motoru validation'dayken üretilen tahmin."""
    X, y_fit = df[INDEX_COLS + FEATURES_14], df["RUL"].clip(upper=cap)
    oof = np.zeros(len(df))
    for tr, va in cv.split(X, groups=df["unit"]):
        m = clone(pipe).fit(X.iloc[tr], y_fit.iloc[tr])
        oof[va] = np.clip(m.predict(X.iloc[va]), 0, None)
    return oof

from sklearn.base import clone
df["pred_LR"] = oof_predictions(baselines["LinearRegression — ham 14 sensör"], df, CAP, cv)
df["RUL_clip"] = df["RUL"].clip(upper=CAP)
horizon = df[df["RUL"] <= 150]

fig, ax = plt.subplots(1, 3, figsize=(19, 5))
ax[0].scatter(horizon["RUL"], horizon["pred_LR"], s=3, alpha=.25)
ax[0].plot([0, 150], [0, 150], "k--", lw=1); ax[0].plot([0, 125, 150], [0, 125, 125], "r-", lw=1, label="hedef (cap=125)")
ax[0].set_xlabel("Gerçek RUL"); ax[0].set_ylabel("Tahmin"); ax[0].set_title("Tahmin vs gerçek (karar ufku)"); ax[0].legend()

res_ = horizon["pred_LR"] - horizon["RUL_clip"]
bins = pd.cut(horizon["RUL"], [0, 25, 50, 75, 100, 125, 150], include_lowest=True)
sns.boxplot(x=bins, y=res_, ax=ax[1], showfliers=False); ax[1].axhline(0, c="k", lw=1)
ax[1].set_xlabel("Gerçek RUL aralığı"); ax[1].set_ylabel("Hata (tahmin − hedef)"); ax[1].set_title("RUL aralığına göre residual")

for un in [3, 24, 81]:
    d = df[df["unit"] == un]
    ax[2].plot(d["cycle"], d["RUL_clip"], "k-", lw=1); ax[2].plot(d["cycle"], d["pred_LR"], lw=1, label=f"motor {un}")
ax[2].set_xlabel("döngü"); ax[2].set_ylabel("RUL"); ax[2].set_title("Örnek motorlar: tahmin (renkli) vs hedef (siyah)"); ax[2].legend()
plt.tight_layout(); plt.show()

**Yorum:**
- Tahminler köşegen etrafında **geniş bir bant** oluşturuyor; tek tek döngülerdeki sensör gürültüsü doğrudan tahmine yansıyor (sağdaki grafikte tahminlerin zikzak çizmesi).
- Residual grafiği sistematik bir desen gösteriyor: model **arızaya yaklaşan motorlarda kalan ömrü fazla tahmin ediyor.** Medyan hata RUL 25–75 aralığında yaklaşık **+17 ile +20 döngü**, RUL < 25'te yaklaşık +9 döngü. Bu pozitif hata, NASA skorunun en ağır cezalandırdığı ve güvenlik açısından en tehlikeli hata türü: motor aslında 30 döngü sonra arızalanacakken model ~50 döngü kaldığını söylüyor. Bozulma ömrün sonuna doğru hızlandığı için doğrusal model bu bölgeyi yakalayamıyor. Sağlıklı bölgede (RUL 100–150) ise tersine, kalan ömrü 10–18 döngü eksik tahmin ediyor.
- Bu iki gözlem, Aşama 8'de **gürültüyü bastıran özellikler + doğrusal olmayan modellerin** neden gerekli olduğunu gösteriyor.

### 7.4 Baseline katsayıları — yorumlanabilirlik

In [ ]:
lr_full = baselines["LinearRegression — ham 14 sensör"].fit(df[INDEX_COLS + FEATURES_14], df["RUL"].clip(upper=CAP))
coef = pd.Series(lr_full.named_steps["model"].coef_, index=FEATURES_14).sort_values()
plt.figure(figsize=(8, 5)); coef.plot(kind="barh", color=["tab:red" if v < 0 else "tab:blue" for v in coef])
plt.title("Linear Regression katsayıları (standartlaştırılmış özellikler)"); plt.xlabel("1 std artışın RUL'a etkisi (döngü)"); plt.show()

**Yorum:** Negatif katsayılı sensörler (`s11` Ps30, `s4` T50, `s15` BPR…) arttıkça kalan ömür azalıyor; pozitif katsayılılar (`s12` phi, `s7` P30, `s20`/`s21` soğutma havası) azaldıkça kalan ömür azalıyor. Bu, Bölüm 1'deki EDA bulgularıyla tutarlı. `s9` ve `s14` çok korelasyonlu olduğu için katsayıları dikkatle yorumlanmalı (aralarında bölünüyor). Bu, Aşama 8'de Ridge'i denemek için bir gerekçe.

### 7.5 Baseline sonuçlarının yorumu

| Model | Test-benzeri RMSE | Yorum |
|---|---|---|
| Dummy (ortalama) | ~44.1 | Alt sınır. R² ≈ −0.15: sabit tahmin, karar ufkunda ortalamadan bile kötü. |
| Dummy (medyan) | ~52.0 | Kırpılmış hedefin medyanı yüksek olduğu için karar ufkunda daha da kötü. |
| **LR — ham 14 sensör (ana baseline)** | **~23.2** | Dummy'ye göre ~%47 daha iyi, R² ≈ 0.67. Literatürde benzer kurulumda raporlanan ~22 RMSE ile tutarlı → **doğrulama protokolümüz güvenilir.** |
| LR — kayan özellikler | ~21.3 | Feature engineering doğrusal modelde ~%8 kazanç sağlıyor. |

**Aşama 8 için çıkarımlar:**
1. Geçilmesi gereken referans: **~23.2 RMSE** (ana baseline). Bölüm 1'deki başarı kriteri (resmi testte ~15–18 RMSE) için baseline'ın en az ~%25 iyileştirilmesi gerekiyor.
2. 6.4'teki duyarlılık kontrolünde, varsayılan HistGradientBoosting kayan özelliklerle zaten ~16.0'a iniyor → doğrusal olmayan modeller ve tuning ile hedefe ulaşmak gerçekçi.
3. Baseline'ın en büyük zayıflığı, arızaya yaklaşan motorlarda (RUL < 75) ömrü sistematik olarak fazla tahmin etmesi. Model seçiminde yalnızca RMSE'ye değil, **NASA skoruna** da bakılacak.

### Sonuçları kaydet

In [ ]:
config = {"features": FEATURES_14, "drop": DROP_COLS, "seed": SEED, "cap": CAP, "window": 30,
          "stats": ["mean", "std", "slope"], "cv": "GroupKFold(n_splits=5)", "eval_max_rul": 150,
          "eval_min_cycle": 31, "eval_n_cuts": 10,
          "baseline_test_RMSE": round(float(base_sum.loc["LinearRegression — ham 14 sensör", "test_RMSE_mean"]), 3)}
json.dump(config, open("project_config.json", "w"), indent=2, ensure_ascii=False)
pd.concat([fe_res, cap_res.drop(columns=["model_ailesi", "cap"]), base_res]).to_csv("results_stage5_7.csv", index=False)
print(json.dumps(config, indent=2, ensure_ascii=False))